# Сравнение моделей с бейзлайном на CIC-IDS2017

In [36]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from cic_common import (
    LIGHT_TO_DARK_BLUE,
    MAIN_COLOR,
    RU,
    label_ru,
    load_dataset,
    show_table,
    style,
    with_spaces,
)

In [37]:
import os
os.chdir(
    '/Users/a.zhukov/dev/phd/notebooks'
)

## 1. Настройки

In [35]:
SAMPLE_SHARE = 1.0
MIN_FLOWS_PER_CLASS = 500
ATTEMPTED_IS_NORMAL = True
VAL_STARTS_AT = 0.6
TEST_STARTS_AT = 0.8
BASELINE_NAME = "Логистическая регрессия"
N_TREES = 200
RANDOM_STATE = 42

## 2. Данные и классы

Класс потока для модели: вид атаки или `BENIGN` для нормального трафика. Попытки атак по совету авторов датасета отнесены к норме (настройка `ATTEMPTED_IS_NORMAL`).

In [38]:
flows = load_dataset()

if not ATTEMPTED_IS_NORMAL:
    flows = flows[flows["Traffic Type"] != "Попытка атаки"]

# Класс потока: вид атаки, а для нормы и попыток атак BENIGN
flows["Class"] = np.where(flows["Traffic Type"] == "Атака", flows["Label"], "BENIGN")

# Момент, когда поток закончился. Длительность записана в микросекундах.
flows["Flow End"] = flows["Timestamp"] + pd.to_timedelta(flows["Flow Duration"], unit="us")


def take_sample(data, share, min_flows_per_class):
    """Оставляет заданную долю потоков каждого класса, но не меньше min_flows_per_class."""
    if share >= 1:
        return data
    kept = []
    for _, class_flows in data.groupby("Class"):
        count = max(round(len(class_flows) * share), min(len(class_flows), min_flows_per_class))
        kept.append(class_flows.sample(count, random_state=RANDOM_STATE))
    return pd.concat(kept)


flows = take_sample(flows, SAMPLE_SHARE, MIN_FLOWS_PER_CLASS)

# В файлах строки идут не строго по времени. Упорядочиваем: дальше время важно.
flows = flows.sort_values("Timestamp").reset_index(drop=True)

print("Потоков в расчёте:", with_spaces(len(flows)))
show_table(flows["Class"].value_counts().rename("Потоков").to_frame())

Потоков в расчёте: 2 099 976


,Потоков
Class,
Норма,1 594 545
Portscan,159 066
DoS Hulk,158 468
DDoS,95 144
Infiltration - Portscan,71 767
DoS GoldenEye,7 567
FTP-Patator,3 972
DoS Slowloris,3 859
SSH-Patator,2 961


## 3. Признаки

Пока только числовые описания поведения потока.

In [ ]:
NOT_FEATURES = {
    "id": "номер строки в файле",
    "Flow ID": "составлен из адресов и портов",
    "Src IP": "адрес стенда: большинство атак идёт с одного адреса",
    "Dst IP": "адрес стенда: большинство атак идёт на один адрес",
    "Src Port": "случайный порт клиента, поведение потока не описывает",
    "Dst Port": "на этом стенде порт выдаёт вид атаки",
    "Timestamp": "каждая атака записана в свои часы",
    "Flow End": "время, служебная колонка",
    "Day": "каждая атака записана в свой день",
    "Label": "ответ",
    "Attack": "ответ",
    "Traffic Type": "ответ",
    "Class": "ответ",
    "Attempted Category": "заполнена по метке, то есть содержит ответ",
    "Average Packet Size": "повторяет Packet Length Mean",
    "Fwd Segment Size Avg": "повторяет Fwd Packet Length Mean",
    "Bwd Segment Size Avg": "повторяет Bwd Packet Length Mean",
    "Packet Length Variance": "квадрат Packet Length Std",
}

feature_columns = [column for column in flows.columns if column not in NOT_FEATURES]

features = flows[feature_columns].copy()

# Два исправления по итогам EDA. Плохие значения заменяем пропусками,
# а пропуски потом заполняет сама модель по обучающей части.
# 1. Бесконечная скорость у потоков с нулевой длительностью.
features = features.replace([np.inf, -np.inf], np.nan)
# 2. Отрицательная длина заголовков.
for column in ["Fwd Header Length", "Bwd Header Length"]:
    features[column] = features[column].where(features[column] >= 0)

# float32 занимает вдвое меньше памяти, чем float64. Моделям такой точности хватает.
X = features.astype("float32")
del features

print("Признаков:", len(feature_columns))
show_table(pd.Series(NOT_FEATURES, name="Почему убрана").rename_axis("Колонка").reset_index())

## Разбиение как в WTMC2021

Здесь повторена схема из [кода авторов исправленной версии датасета](https://github.com/GintsEngelen/WTMC2021-Code), файлы `MakeDataNumpyFriendly.py` и `Benchmarking_RF.py`:

- потоки `Attempted` считаются `BENIGN`;
- адреса, порты, идентификатор потока и время в признаки не входят;
- строки с NaN и бесконечностями удаляются;
- разбиение случайное, со стратификацией по классу: 75% train, 25% test, `random_state=44`;
- случайный лес: 50 деревьев, глубина не больше 20;
- метрики: weighted precision, weighted recall, weighted F1.

Отличия от авторов:
- признаки взяты из раздела 3, то есть без четырёх признаков, которые повторяют другие;
- кроме случайного леса обучаются логистическая регрессия, наивный байесовский классификатор, LightGBM и CatBoost;
- к метрикам авторов добавлены accuracy, macro F1 и бинарные метрики.

Разбиение случайное: время потоков здесь не учитывается.

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

WTMC_TEST_SIZE = 0.25
WTMC_RANDOM_STATE = 44

# Берём значения признаков как есть, без исправлений из раздела 3:
# авторы ничего не исправляют, а только удаляют строки, где есть NaN или бесконечность.
wtmc_X = flows[feature_columns].astype("float32")
is_finite_row = np.isfinite(wtmc_X).all(axis=1).to_numpy()
wtmc_X = wtmc_X[is_finite_row]

# Модели работают с номерами классов. Номер класса это его место в списке wtmc_class_names.
wtmc_y, wtmc_class_names = pd.factorize(flows.loc[is_finite_row, "Class"], sort=True)

wtmc_X_train, wtmc_X_test, wtmc_y_train, wtmc_y_test = train_test_split(
    wtmc_X,
    wtmc_y,
    test_size=WTMC_TEST_SIZE,
    stratify=wtmc_y,
    random_state=WTMC_RANDOM_STATE,
)

print("Удалено строк с NaN или бесконечностью:", (~is_finite_row).sum())
print("Train:", with_spaces(len(wtmc_X_train)), "Test:", with_spaces(len(wtmc_X_test)))

# Сколько потоков каждого класса в train и в test
class_counts = pd.DataFrame(
    {"Train": np.bincount(wtmc_y_train), "Test": np.bincount(wtmc_y_test)},
    index=wtmc_class_names,
)
show_table(class_counts.sort_values("Train", ascending=False))

In [ ]:
def signed_log(values):
    """Сжимает разброс значений: log(1 + |x|) с сохранением знака.

    Такая же функция ещё раз объявлена в разделе 5.
    """
    return np.sign(values) * np.log1p(np.abs(values))


wtmc_models = {
    "DummyClassifier": DummyClassifier(strategy="most_frequent"),
    # Логистической регрессии и байесовскому классификатору важен масштаб признаков:
    # сжимаем разброс логарифмом и приводим признаки к общему масштабу.
    "Логистическая регрессия": make_pipeline(
        FunctionTransformer(signed_log), StandardScaler(), LogisticRegression(max_iter=300)
    ),
    "Наивный байесовский классификатор": make_pipeline(
        FunctionTransformer(signed_log), StandardScaler(), GaussianNB()
    ),
    # Параметры случайного леса как у авторов
    "Случайный лес (WTMC2021)": RandomForestClassifier(
        n_estimators=50, max_depth=20, n_jobs=-1, random_state=WTMC_RANDOM_STATE
    ),
    # У LightGBM по умолчанию нет ограничения на величину поправки в листе дерева.
    # На классах из нескольких потоков обучение из-за этого расходится, поэтому ограничения заданы явно.
    "LightGBM": LGBMClassifier(
        n_estimators=N_TREES,
        reg_lambda=1.0,
        min_child_weight=1.0,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=N_TREES, random_seed=RANDOM_STATE, verbose=0, allow_writing_files=False
    ),
}

wtmc_predictions, wtmc_fit_seconds = {}, {}
for name, model in wtmc_models.items():
    started = time.perf_counter()
    model.fit(wtmc_X_train, wtmc_y_train)
    wtmc_fit_seconds[name] = time.perf_counter() - started
    # ravel нужен для CatBoost: он возвращает предсказания столбцом, а не строкой
    wtmc_predictions[name] = np.asarray(model.predict(wtmc_X_test)).ravel()
    print(f"{name}: fit {wtmc_fit_seconds[name]:.1f} с")

### Метрики на test

**Многоклассовые метрики.** Классы: `BENIGN` и каждый вид атаки отдельно.
- *Weighted Precision, Weighted Recall, Weighted F1*: метрика считается для каждого класса, потом берётся среднее с весами по числу потоков класса. Так считают авторы. Почти весь вес приходится на `BENIGN`.
- *Macro F1*: среднее F1 по классам без весов. Редкая атака весит столько же, сколько `BENIGN`.
- *Accuracy*: доля верно классифицированных потоков. Всегда равна Weighted Recall.

In [ ]:
wtmc_rows = {}
for name, y_pred in wtmc_predictions.items():
    precision, recall, f1, _ = precision_recall_fscore_support(
        wtmc_y_test, y_pred, average="weighted", zero_division=0
    )
    wtmc_rows[name] = {
        "Weighted Precision": precision,
        "Weighted Recall": recall,
        "Weighted F1": f1,
        "Macro F1": f1_score(wtmc_y_test, y_pred, average="macro", zero_division=0),
        "Accuracy": accuracy_score(wtmc_y_test, y_pred),
        "Fit time, с": wtmc_fit_seconds[name],
    }

wtmc_scores = pd.DataFrame.from_dict(wtmc_rows, orient="index")
show_table(wtmc_scores, digits=4)

**Бинарные метрики.** Positive это любая атака, negative это `BENIGN`. Вид атаки не учитывается. FPR = FP / (FP + TN).

In [ ]:
wtmc_benign_code = list(wtmc_class_names).index("BENIGN")
is_attack = wtmc_y_test != wtmc_benign_code

wtmc_binary_rows = {}
for name, y_pred in wtmc_predictions.items():
    predicted_attack = y_pred != wtmc_benign_code
    precision, recall, f1, _ = precision_recall_fscore_support(
        is_attack, predicted_attack, average="binary", zero_division=0
    )
    false_positives = (~is_attack & predicted_attack).sum()
    wtmc_binary_rows[name] = {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": false_positives / (~is_attack).sum(),
    }

show_table(pd.DataFrame.from_dict(wtmc_binary_rows, orient="index"), digits=4)

**F1 по классам.** Support это число потоков класса в test. Чем оно меньше, тем менее надёжна метрика.

In [ ]:
all_class_codes = np.arange(len(wtmc_class_names))

f1_by_class = pd.DataFrame(
    {
        name: f1_score(wtmc_y_test, y_pred, labels=all_class_codes, average=None, zero_division=0)
        for name, y_pred in wtmc_predictions.items()
    },
    index=wtmc_class_names,
)
f1_by_class.insert(0, "Support", np.bincount(wtmc_y_test, minlength=len(wtmc_class_names)))
show_table(f1_by_class.sort_values("Support", ascending=False), digits=4)

## 4. Разбиение на обучение, валидацию и тест

In [ ]:
TRAIN, VAL, TEST = "обучение", "валидация", "тест"
DROPPED = "удалён на границе"


def find_attack_episodes(flows):
    """Находит эпизоды атак и границы частей внутри каждого эпизода.

    Эпизод: промежуток от первого до последнего пакета атаки.
    Если две атаки пересекаются по времени, они образуют один эпизод.
    """
    attack_flows = flows[flows["Traffic Type"] == "Атака"]
    windows = (
        attack_flows.groupby("Label")
        .agg(start=("Timestamp", "min"), end=("Flow End", "max"))
        .sort_values("start")
    )
    # Дальше считаем, что правая граница в эпизод не входит. Сдвигаем её на 1 мкс,
    # чтобы последний пакет атаки остался внутри эпизода.
    windows["end"] += pd.Timedelta(microseconds=1)

    episodes = []
    for label, window in windows.iterrows():
        overlaps_previous = bool(episodes) and window["start"] < episodes[-1]["end"]
        if overlaps_previous:
            episodes[-1]["labels"].append(label)
            episodes[-1]["end"] = max(episodes[-1]["end"], window["end"])
        else:
            episodes.append({"labels": [label], "start": window["start"], "end": window["end"]})

    # Границы валидации и теста: моменты, к которым началась заданная доля потоков атаки
    for episode in episodes:
        attack_times = attack_flows.loc[attack_flows["Label"].isin(episode["labels"]), "Timestamp"]
        episode["val_start"] = attack_times.quantile(VAL_STARTS_AT)
        episode["test_start"] = attack_times.quantile(TEST_STARTS_AT)

    return pd.DataFrame(episodes)


def find_quiet_cuts(flows, episodes):
    """Для каждого дня находит границы валидации и теста во времени без атак.

    Возвращает таблицу, где строка это день (его полночь).
    """
    during_attack = np.zeros(len(flows), dtype=bool)
    for episode in episodes.itertuples():
        during_attack |= flows["Timestamp"].between(episode.start, episode.end, inclusive="left").to_numpy()

    quiet_times = flows.loc[~during_attack, "Timestamp"]
    cuts = quiet_times.groupby(quiet_times.dt.normalize()).quantile([VAL_STARTS_AT, TEST_STARTS_AT]).unstack()
    cuts.columns = ["val_start", "test_start"]
    return cuts


def part_at(moment, episodes, quiet_cuts):
    """Определяет, к какой части относится момент времени."""
    current_episodes = episodes[(episodes["start"] <= moment) & (moment < episodes["end"])]
    if len(current_episodes) > 0:
        cuts = current_episodes.iloc[0]  # идёт атака: действуют границы эпизода
    else:
        cuts = quiet_cuts.loc[moment.normalize()]  # атак нет: действуют границы дня

    if moment < cuts["val_start"]:
        return TRAIN
    if moment < cuts["test_start"]:
        return VAL
    return TEST


def build_timeline(episodes, quiet_cuts):
    """Собирает расписание: когда начинается каждый отрезок времени и к какой части он относится.

    Отрезок длится до начала следующего отрезка.
    """
    moments = sorted(
        set(quiet_cuts.index)  # начало каждого дня
        | set(quiet_cuts["val_start"])
        | set(quiet_cuts["test_start"])
        | set(episodes["start"])
        | set(episodes["val_start"])
        | set(episodes["test_start"])
        | set(episodes["end"])
    )
    timeline = pd.DataFrame({"start": moments})
    timeline["part"] = [part_at(moment, episodes, quiet_cuts) for moment in moments]
    # Номер куска растёт на единицу каждый раз, когда одна часть сменяется другой.
    # Соседние отрезки одной части получают один номер.
    timeline["piece"] = (timeline["part"] != timeline["part"].shift()).cumsum()
    return timeline


def assign_parts(flows, timeline):
    """Определяет часть для каждого потока.

    Поток относится к той части, в которой он начался.
    Если он закончился уже в другой части, поток помечается как удалённый.
    """
    block_starts = timeline["start"].to_numpy()
    block_of_start = np.searchsorted(block_starts, flows["Timestamp"].to_numpy(), side="right") - 1
    block_of_end = np.searchsorted(block_starts, flows["Flow End"].to_numpy(), side="right") - 1

    part = timeline["part"].to_numpy()[block_of_start]
    piece = timeline["piece"].to_numpy()
    crosses_border = piece[block_of_start] != piece[block_of_end]
    return np.where(crosses_border, DROPPED, part)


episodes = find_attack_episodes(flows)
quiet_cuts = find_quiet_cuts(flows, episodes)
timeline = build_timeline(episodes, quiet_cuts)
flows["Part"] = assign_parts(flows, timeline)

### Проверка разбиения

Эпизоды атак и границы частей внутри них.

In [ ]:
episodes_table = pd.DataFrame(
    {
        "Атаки": episodes["labels"].str.join(", "),
        "День": episodes["start"].dt.strftime("%d.%m"),
        "Начало, обучение": episodes["start"].dt.strftime("%H:%M:%S"),
        "Начало валидации": episodes["val_start"].dt.strftime("%H:%M:%S"),
        "Начало теста": episodes["test_start"].dt.strftime("%H:%M:%S"),
        "Конец": episodes["end"].dt.strftime("%H:%M:%S"),
    }
)
show_table(episodes_table)

In [ ]:
PART_ORDER = [TRAIN, VAL, TEST, DROPPED]

part_sizes = flows["Part"].value_counts().reindex(PART_ORDER, fill_value=0).rename("Потоков").to_frame()
part_sizes["Доля, %"] = part_sizes["Потоков"] / len(flows) * 100
show_table(part_sizes)

In [ ]:
def count_time_overlaps(flows):
    """Считает потоки, которые начались раньше, чем закончился более ранний поток из другой части."""
    kept = flows[flows["Part"] != DROPPED].sort_values("Timestamp")
    overlaps = 0
    for part in [TRAIN, VAL, TEST]:
        # Для каждого потока: самый поздний конец среди потоков этой части, которые начались до него
        ends_of_part = kept["Flow End"].where(kept["Part"] == part)
        latest_end_before = ends_of_part.cummax().ffill().shift()
        overlaps += ((kept["Part"] != part) & (kept["Timestamp"] < latest_end_before)).sum()
    return overlaps


print("Потоков, которые идут одновременно с потоком из другой части:", count_time_overlaps(flows))

In [ ]:
# Сколько потоков каждого класса попало в каждую часть
class_parts = pd.crosstab(flows["Class"], flows["Part"]).reindex(columns=PART_ORDER, fill_value=0)
show_table(class_parts.sort_values(TRAIN, ascending=False))

In [ ]:
# Классы, у которых в валидации или тесте меньше 30 потоков: оценки для них ненадёжны
FEW_FLOWS = 30
rare_classes = class_parts[(class_parts[VAL] < FEW_FLOWS) | (class_parts[TEST] < FEW_FLOWS)]
show_table(rare_classes[[TRAIN, VAL, TEST]])

### Расписание частей

Каждая полоса показывает один день. Цвет отрезка показывает, в какую часть попадают потоки, начавшиеся в это время.

In [ ]:
PART_COLORS = {TRAIN: "#2a78d6", VAL: "#eb6834", TEST: "#1baf7a"}


def hours_from_midnight(moments):
    """Переводит моменты времени в часы от начала суток: 13:30 -> 13.5."""
    return (moments - moments.dt.normalize()).dt.total_seconds() / 3600


def describe_block(start, episodes):
    """Подпись отрезка: названия атак, если в это время идёт эпизод, иначе 'без атак'."""
    current_episodes = episodes[(episodes["start"] <= start) & (start < episodes["end"])]
    if len(current_episodes) > 0:
        return ", ".join(current_episodes.iloc[0]["labels"])
    return "без атак"


# Отрезки расписания с началом и концом. Конец отрезка совпадает с началом следующего.
blocks = timeline.copy()
blocks["end"] = blocks["start"].shift(-1)

# Запись идёт не круглые сутки. Обрезаем отрезки по первому и последнему потоку дня.
day_of_flow = flows["Timestamp"].dt.normalize()
first_flow = flows.groupby(day_of_flow)["Timestamp"].min()
last_flow = flows.groupby(day_of_flow)["Flow End"].max()
block_day = blocks["start"].dt.normalize()
blocks["start"] = np.maximum(blocks["start"], block_day.map(first_flow))
blocks["end"] = np.minimum(blocks["end"].fillna(blocks["start"].max()), block_day.map(last_flow))
blocks = blocks[blocks["end"] > blocks["start"]]

day_names = flows.groupby(day_of_flow)["Day"].first().astype(str)
blocks["day"] = block_day.map(day_names) + blocks["start"].dt.strftime(", %d.%m")
blocks["start_hour"] = hours_from_midnight(blocks["start"])
blocks["length_hours"] = (blocks["end"] - blocks["start"]).dt.total_seconds() / 3600
blocks["hint"] = [
    f"{start:%H:%M:%S} - {end:%H:%M:%S}<br>{describe_block(start, episodes)}"
    for start, end in zip(blocks["start"], blocks["end"])
]

fig = go.Figure()
for part, color in PART_COLORS.items():
    part_blocks = blocks[blocks["part"] == part]
    fig.add_trace(
        go.Bar(
            name=part,
            orientation="h",
            y=part_blocks["day"],
            base=part_blocks["start_hour"],  # где отрезок начинается
            x=part_blocks["length_hours"],  # сколько он длится
            marker_color=color,
            text=part_blocks["hint"],
            textposition="none",
            hovertemplate="%{text}<extra>" + part + "</extra>",
        )
    )
style(
    fig,
    title="Какой части принадлежит каждый отрезок времени<br><sup>Узкие тройки отрезков это эпизоды атак</sup>",
    x_title="Время суток, часы (UTC)",
    height=420,
)
fig.update_layout(barmode="overlay", bargap=0.45)
fig.update_xaxes(dtick=1, ticksuffix=":00")
fig.update_yaxes(autorange="reversed", title_text="День")
fig.show()

### train test val

In [ ]:
# Модели работают с номерами классов. Номер класса это его место в списке class_names.
class_names = np.array(sorted(flows["Class"].unique()))
y = pd.Categorical(flows["Class"], categories=class_names).codes
NORMAL_CODE = list(class_names).index("BENIGN")

is_train = (flows["Part"] == TRAIN).to_numpy()
is_val = (flows["Part"] == VAL).to_numpy()
is_test = (flows["Part"] == TEST).to_numpy()

# Модель не может научиться классу, которого нет в обучении
classes_without_training = set(class_names) - set(flows.loc[is_train, "Class"])
assert not classes_without_training, f"В обучении нет классов: {classes_without_training}"

X_train, y_train = X[is_train], y[is_train]
X_val, y_val = X[is_val], y[is_val]
X_test, y_test = X[is_test], y[is_test]

print("Обучение:", with_spaces(len(X_train)), "Валидация:", with_spaces(len(X_val)), "Тест:", with_spaces(len(X_test)))

## 5. Модели

In [ ]:
def signed_log(values):
    """Сжимает разброс значений: log(1 + |x|) с сохранением знака."""
    return np.sign(values) * np.log1p(np.abs(values))


def with_scaling(model):
    """Подготовка для моделей, которым важен масштаб признаков.

    Заполняем пропуски, сжимаем разброс логарифмом, приводим признаки к общему масштабу.
    """
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(signed_log),
        StandardScaler(),
        model,
    )


def with_filled_gaps(model):
    """Подготовка для дерева и случайного леса: только заполняем пропуски."""
    return make_pipeline(SimpleImputer(strategy="median"), model)


def make_models():
    """Создаёт необученные модели. Ключ словаря это название модели в таблицах и на графиках."""
    return {
        "DummyClassifier": DummyClassifier(strategy="most_frequent"),
        "Логистическая регрессия": with_scaling(LogisticRegression(max_iter=300)),
        "Наивный байесовский классификатор": with_scaling(GaussianNB()),
        "Дерево решений": with_filled_gaps(DecisionTreeClassifier(random_state=RANDOM_STATE)),
        "Случайный лес": with_filled_gaps(
            RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
        ),
        # Бустинги работают с пропусками сами, подготовка им не нужна.
        # У HistGradientBoosting и у LightGBM по умолчанию нет ограничения на величину поправки в листе дерева.
        # На классах из нескольких потоков обучение из-за этого расходится, и модель отвечает почти наугад.
        # Поэтому им заданы те же ограничения, которые в XGBoost стоят по умолчанию.
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=N_TREES, l2_regularization=1.0, random_state=RANDOM_STATE
        ),
        "XGBoost": XGBClassifier(n_estimators=N_TREES, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
        "LightGBM": LGBMClassifier(
            n_estimators=N_TREES,
            reg_lambda=1.0,
            min_child_weight=1.0,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=N_TREES, random_seed=RANDOM_STATE, verbose=0, allow_writing_files=False
        ),
    }

## 6. Обучение

Каждая модель учится на обучающей части и делает предсказания для валидации и теста. Предсказания сохраняются, чтобы потом считать по ним любые метрики без повторного обучения.

In [ ]:
def fit_and_predict(models, X_train, y_train, parts_to_predict):
    """Обучает модели по очереди и сохраняет их ответы.

    parts_to_predict: словарь «название части -> признаки потоков этой части».
    Возвращает ответы в виде «название модели -> название части -> ответы»
    и таблицу со временем работы каждой модели.
    """
    predictions, timings = {}, {}
    for name, model in models.items():
        started = time.perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - started

        started = time.perf_counter()
        # ravel нужен для CatBoost: он возвращает ответы столбцом, а не строкой
        predictions[name] = {
            part: np.asarray(model.predict(part_features)).ravel()
            for part, part_features in parts_to_predict.items()
        }
        predict_seconds = time.perf_counter() - started

        timings[name] = {"Fit time, с": fit_seconds, "Predict time, с": predict_seconds}
        print(f"{name}: fit {fit_seconds:.1f} с, predict {predict_seconds:.1f} с")

    return predictions, pd.DataFrame.from_dict(timings, orient="index")


models = make_models()
predictions, timings = fit_and_predict(models, X_train, y_train, {VAL: X_val, TEST: X_test})

## 7. Сравнение на валидации


**Многоклассовые метрики.** Классы: `BENIGN` и каждый вид атаки отдельно.
- *Macro F1*: F1 считается для каждого класса отдельно, потом берётся невзвешенное среднее. Редкая атака весит столько же, сколько `BENIGN`, поэтому пропуск редких атак сильно снижает метрику. Это основная метрика для сравнения.

Все метрики в таблицах даны в долях от 0 до 1.

In [34]:
def ratio(count, total):
    """Доля от 0 до 1. Если делить не на что, возвращает 0."""
    return count / total if total > 0 else 0.0


def binary_metrics(is_attack, predicted_attack):
    """Precision, Recall, F1 и FPR для бинарной задачи: positive это атака, negative это BENIGN."""
    tp = (is_attack & predicted_attack).sum()
    fp = (~is_attack & predicted_attack).sum()
    fn = (is_attack & ~predicted_attack).sum()
    tn = (~is_attack & ~predicted_attack).sum()

    precision = ratio(tp, tp + fp)
    recall = ratio(tp, tp + fn)
    return {
        "Precision": precision,
        "Recall": recall,
        "F1": ratio(2 * precision * recall, precision + recall),
        "FPR": ratio(fp, fp + tn),
    }


def comparison_table(predictions, y_true, part, normal_code):
    """Таблица метрик: строка это модель, столбец это метрика."""
    # В macro F1 входят только классы, которые есть в этой части
    classes_in_part = np.unique(y_true)
    rows = {}
    for name, model_predictions in predictions.items():
        y_pred = model_predictions[part]
        rows[name] = {
            "Macro F1": f1_score(y_true, y_pred, labels=classes_in_part, average="macro", zero_division=0),
            **binary_metrics(y_true != normal_code, y_pred != normal_code),
            "Accuracy": ratio((y_true == y_pred).sum(), len(y_true)),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


val_scores = comparison_table(predictions, y_val, VAL, NORMAL_CODE)
show_table(val_scores.join(timings), digits=4)

,Macro F1,Precision,Recall,F1,FPR,Accuracy,"Обучение, с","Ответы, с"
Пустая модель,"0,0529","0,0000","0,0000","0,0000","0,0000","0,7332","0,0336","0,0005"
Логистическая регрессия,"0,7612","0,9977","0,9986","0,9982","0,0008","0,9892","101,7996","0,9267"
Наивный байесовский классификатор,"0,7052","0,5336","0,9972","0,6952","0,3172","0,7545","7,3382","2,9718"
Дерево решений,"0,8707","0,9952","0,9995","0,9974","0,0017","0,9932","56,1500","0,3448"
Случайный лес,"0,8675","1,0000","0,9998","0,9999","0,0000","0,9945","38,0342","1,1403"
Бустинг scikit-learn,"0,9722","0,9998","0,9999","0,9998","0,0001","0,9956","143,9658","6,4765"
XGBoost,"0,9084","0,9999","0,9999","0,9999","0,0000","0,9945","38,1355","0,7249"
LightGBM,"0,8922","0,9997","0,9999","0,9998","0,0001","0,9945","26,0705","3,7475"
CatBoost,"0,5140","0,9949","0,9922","0,9935","0,0019","0,9875","278,9624","0,1914"


### Разница с бейзлайном

Из метрик каждой модели вычтены метрики бейзлайна. Положительное число означает, что у модели метрика выше. Для FPR лучше отрицательное число.

In [ ]:
show_table(val_scores - val_scores.loc[BASELINE_NAME], digits=4)

In [ ]:
MAIN_SCORE = "Macro F1"

# Лучшая модель должна оказаться наверху: на горизонтальном графике первая строка рисуется внизу
chart_data = val_scores[[MAIN_SCORE]].sort_values(MAIN_SCORE)

fig = chart_data.iplot(kind="barh", colors=MAIN_COLOR, opacity=1, asFigure=True)
style(
    fig,
    title=f"{MAIN_SCORE}, валидация<br><sup>Вертикальная линия показывает уровень бейзлайна</sup>",
    x_title=MAIN_SCORE,
    height=460,
)
fig.add_vline(x=val_scores.loc[BASELINE_NAME, MAIN_SCORE], line_color="#0b0b0b", line_width=1)
fig.update_xaxes(range=[0, 1])
fig.update_layout(showlegend=False)
fig.show()

### Recall по классам

Recall каждого класса для каждой модели, в процентах. В скобках указан support: число потоков класса в валидации. Чем их меньше, тем менее надёжна оценка.

In [ ]:
def recall_by_class(predictions, y_true, part):
    """Recall каждого класса в процентах. Строка это модель, столбец это класс."""
    true_class = class_names[y_true]
    table = {}
    for name, model_predictions in predictions.items():
        is_correct = pd.Series(model_predictions[part] == y_true)
        table[name] = is_correct.groupby(true_class).mean() * 100
    return pd.DataFrame(table).T


def plot_heatmap(table, title, x_title, y_title, height):
    """Рисует таблицу долей от 0 до 100% как тепловую карту с числами в клетках."""
    fig = go.Figure(
        go.Heatmap(
            z=table.to_numpy(),
            x=list(table.columns),
            y=list(table.index),
            colorscale=LIGHT_TO_DARK_BLUE,
            zmin=0,
            zmax=100,
            xgap=2,
            ygap=2,
            texttemplate="%{z:.0f}",
            colorbar=dict(title="%"),
            hovertemplate="%{y}<br>%{x}<br>%{z:.1f}%<extra></extra>",
        )
    )
    style(fig, title=title, x_title=x_title, y_title=y_title, height=height)
    fig.update_layout(margin_l=270)  # слева нужно место для длинных названий
    fig.update_yaxes(autorange="reversed")
    fig.show()


def class_captions(y_true):
    """Подписи классов с числом потоков: 'DoS Hulk (31 694)'."""
    counts = pd.Series(class_names[y_true]).value_counts()
    return {name: f"{label_ru(name)} ({with_spaces(count)})" for name, count in counts.items()}


val_recall = recall_by_class(predictions, y_val, VAL).rename(columns=class_captions(y_val))
plot_heatmap(
    val_recall,
    title="Recall по классам, валидация, %",
    x_title="Класс и support в валидации",
    y_title=None,
    height=560,
)

### Confusion matrix лучшей модели

Строка это истинный класс, столбец это предсказанный класс. Значения нормированы по строке и даны в процентах, поэтому на диагонали стоит recall класса.

In [ ]:
best_model_name = val_scores[MAIN_SCORE].idxmax()
print("Лучшая модель на валидации:", best_model_name)

all_classes = np.arange(len(class_names))
confusion = confusion_matrix(y_val, predictions[best_model_name][VAL], labels=all_classes, normalize="true")
class_captions_ru = [label_ru(name) for name in class_names]

plot_heatmap(
    pd.DataFrame(confusion * 100, index=class_captions_ru, columns=class_captions_ru),
    title=f"Confusion matrix модели «{best_model_name}», валидация, % от истинного класса",
    x_title="Предсказанный класс",
    y_title="Истинный класс",
    height=700,
)

### Feature importance

Feature importance случайного леса (среднее уменьшение Gini impurity). Все значения в сумме дают 1.

In [ ]:
forest = models["Случайный лес"][-1]  # последний шаг цепочки это сама модель
importance = pd.Series(forest.feature_importances_, index=[RU[column] for column in feature_columns])

chart_data = importance.sort_values().tail(20).rename("Feature importance").to_frame()
fig = chart_data.iplot(kind="barh", colors=MAIN_COLOR, opacity=1, asFigure=True)
style(fig, title="Feature importance случайного леса: 20 самых важных признаков", x_title="Feature importance", height=640)
fig.update_layout(showlegend=False)
fig.show()

## 8. Итоговая проверка на тесте

Те же метрики на тестовой части. Модели учились только на обучающей части и теста не видели.

In [ ]:
test_scores = comparison_table(predictions, y_test, TEST, NORMAL_CODE)
show_table(test_scores, digits=4)

In [ ]:
show_table(test_scores - test_scores.loc[BASELINE_NAME], digits=4)

In [ ]:
test_recall = recall_by_class(predictions, y_test, TEST).rename(columns=class_captions(y_test))
plot_heatmap(
    test_recall,
    title="Recall по классам, тест, %",
    x_title="Класс и support в тесте",
    y_title=None,
    height=560,
)

## 9. Строгая проверка: обучение на прошлых днях, проверка на будущих

Здесь время соблюдается полностью: модель учится на понедельнике, вторнике и среде, валидация идёт на четверге, тест на пятнице. Ни один поток обучения не записан позже проверочных.

Плата за строгость: атаки четверга и пятницы в обучении не встречались. Поэтому задача здесь бинарная: модель отвечает только `BENIGN` или «атака». Проверяется, замечает ли модель атаки, которых раньше не видела. Это намного труднее, чем узнавать продолжение знакомой атаки, и метрики здесь ожидаемо ниже, чем в разделах 7 и 8.

In [ ]:
PART_OF_DAY = {
    "Понедельник": TRAIN,
    "Вторник": TRAIN,
    "Среда": TRAIN,
    "Четверг": VAL,
    "Пятница": TEST,
}
day_part = flows["Day"].astype(str).map(PART_OF_DAY).to_numpy()

# Ответ для модели: 1 для атаки, 0 для нормы
is_attack = (flows["Class"] != "BENIGN").to_numpy().astype(int)
NORMAL = 0

day_models = make_models()
day_predictions, day_timings = fit_and_predict(
    day_models,
    X[day_part == TRAIN],
    is_attack[day_part == TRAIN],
    {VAL: X[day_part == VAL], TEST: X[day_part == TEST]},
)

In [ ]:
def binary_table(predictions, is_attack, part):
    """Precision, Recall, F1 и FPR для каждой модели."""
    rows = {
        name: binary_metrics(is_attack == 1, model_predictions[part] != NORMAL)
        for name, model_predictions in predictions.items()
    }
    return pd.DataFrame.from_dict(rows, orient="index")


print("Валидация: четверг")
show_table(binary_table(day_predictions, is_attack[day_part == VAL], VAL), digits=4)

print("Тест: пятница")
show_table(binary_table(day_predictions, is_attack[day_part == TEST], TEST), digits=4)

### Recall по атакам, которых не было в обучении

Для каждой модели и каждой атаки четверга и пятницы: recall, то есть доля потоков атаки, предсказанных как атака. В скобках указан support.

In [ ]:
def recall_by_attack(predictions, part):
    """Recall каждой атаки в процентах. Строка это модель, столбец это атака."""
    part_flows = flows[day_part == part]
    attack_rows = (part_flows["Class"] != "BENIGN").to_numpy()
    attack_names = part_flows.loc[attack_rows, "Class"].to_numpy()

    counts = pd.Series(attack_names).value_counts()
    captions = {name: f"{name} ({with_spaces(count)})" for name, count in counts.items()}

    table = {}
    for name, model_predictions in predictions.items():
        predicted_attack = pd.Series(model_predictions[part][attack_rows] != NORMAL)
        table[name] = predicted_attack.groupby(attack_names).mean() * 100
    return pd.DataFrame(table).T.rename(columns=captions)


attack_recall = recall_by_attack(day_predictions, VAL).join(recall_by_attack(day_predictions, TEST))
plot_heatmap(
    attack_recall,
    title="Recall по атакам, которых не было в обучении, %",
    x_title="Атака и support (четверг и пятница)",
    y_title=None,
    height=520,
)

### Почему нельзя делить случайно

Потоки записаны подряд во времени, и соседние потоки похожи друг на друга: одна атака порождает тысячи почти одинаковых потоков за несколько минут. При случайном разбиении такие соседи попадают и в обучение, и в тест. Модель видит на проверке почти то же, что видела при обучении, и оценка качества получается завышенной. Кроме того, модель учится на потоках, которые в жизни появились бы позже проверочных.

### Почему нельзя просто разрезать неделю по времени

Привычный способ для временных рядов: начало ряда в обучение, середина в валидацию, конец в тест. Здесь он не подходит для распознавания видов атак. По данным EDA каждый вид атаки записан один раз, в один день и в один промежуток времени. Если отдать в тест пятницу, то Botnet, Portscan и DDoS окажутся только в тесте, и модель не сможет научиться их называть. Такое разбиение проверяет другое умение: замечать атаки, которых модель раньше не видела. Оно сделано отдельно в разделе 9.

### Что сделано

Время разрезано на отрезки, и каждый отрезок целиком отдан одной части.

1. **Эпизод атаки.** Это промежуток от первого до последнего пакета одной атаки. Внутри эпизода время идёт так: сначала обучение, потом валидация, потом тест. Границы поставлены в моменты, к которым началось 60% и 80% потоков атаки. Нормальные потоки, которые шли во время эпизода, попадают в ту же часть, что и атака в этот момент.
2. **Время без атак.** Для каждого дня отдельно: первые 60% потоков в обучение, следующие 20% в валидацию, последние 20% в тест.
3. **Потоки на границах.** Поток, который начался в одной части, а закончился в другой, удаляется. Таких потоков меньше 1%.

В похожем решении с GitHub (ссылка в начале блокнота) каждая метка делится по времени отдельно. Тогда нормальные потоки валидации могут идти в те же минуты, что и потоки атаки из обучения. Здесь границы общие для всех потоков, поэтому такого не бывает.

### Что это даёт

- Внутри каждого эпизода и каждого дня модель учится на более раннем трафике, а проверяется на более позднем.
- Любой момент времени принадлежит только одной части. После удаления потоков на границах ни один поток обучения не идёт одновременно с потоком валидации или теста.
- Каждый вид атаки есть во всех трёх частях, поэтому можно оценивать распознавание видов атак.
- Нормальные потоки, которые шли во время атаки, проверяются вместе с ней. Это самые трудные нормальные потоки: атака на сервер меняет и поведение обычных соединений с ним.

### Чего это не даёт

- Тест остаётся продолжением того же эпизода, на начале которого модель училась: тот же атакующий, та же цель, та же программа. Оценка показывает, узнаёт ли модель продолжение знакомой атаки. Она ничего не говорит о новой записи той же атаки в другой сети. Для такой проверки в датасете нет данных.
- У редких атак в валидации и тесте оказывается по несколько потоков. Оценки для них ненадёжны: один поток меняет результат на десятки процентов.
- У медленных атак потоки длятся долго, и на границах теряется заметная их часть: у DoS Slowhttptest около трети потоков, у DoS Slowloris около 12%.